# (1) Setup & Imports

In [1]:
# Install libraries if needed
!pip install requests beautifulsoup4 pandas


# Imports
import requests
from bs4 import BeautifulSoup
import pandas as pd
import sqlite3





# (2) Scraping Books

In [2]:
# Example: scrape first 5 pages of "All products"
base_url = "http://books.toscrape.com/"

# Get category links (first 3 categories for demo)
res = requests.get(base_url)
soup = BeautifulSoup(res.text, "html.parser")

categories = []
for cat in soup.select(".side_categories ul li ul li a")[:3]:
    cat_name = cat.text.strip()
    cat_link = base_url + cat["href"]
    categories.append((cat_name, cat_link))

books = []

# Scrape each category with pagination
for cat_name, cat_link in categories:
    res = requests.get(cat_link)
    soup = BeautifulSoup(res.text, "html.parser")

    while True:
        for item in soup.select(".product_pod"):
            title = item.h3.a["title"]
            price = item.select_one(".price_color").text
            rating = item.p["class"][1]
            availability = item.select_one(".availability").text.strip()

            books.append([title, price, rating, availability, cat_name])

        next_page = soup.select_one("li.next a")
        if next_page:
            next_url = cat_link.replace("index.html", "") + next_page["href"]
            res = requests.get(next_url)
            soup = BeautifulSoup(res.text, "html.parser")
        else:
            break

df = pd.DataFrame(books, columns=["title","price","star_rating","availability","category"])
print("Total books scraped:", len(df))
df.head()






Total books scraped: 69


,title,price,star_rating,availability,category
0,It's Only the Himalayas,Â£45.17,Two,In stock,Travel
1,Full Moon over Noahâs Ark: An Odyssey to Mou...,Â£49.43,Four,In stock,Travel
2,See America: A Celebration of Our National Par...,Â£48.87,Three,In stock,Travel
3,Vagabonding: An Uncommon Guide to the Art of L...,Â£36.94,Two,In stock,Travel
4,Under the Tuscan Sun,Â£37.33,Three,In stock,Travel


# (3) Cleaning Data

In [3]:
# Price → float
df["price_clean"] = df["price"].str.replace("£","", regex=False).str.strip()
df["price_gbp"] = pd.to_numeric(df["price_clean"], errors="coerce")
median_price = df["price_gbp"].median()
df["price_gbp"].fillna(median_price, inplace=True)

# Rating → integer
rating_map = {"One":1,"Two":2,"Three":3,"Four":4,"Five":5}
df["rating"] = df["star_rating"].map(rating_map)

# Availability → boolean
df["in_stock"] = df["availability"].str.contains("In stock")

# Currency conversion
GBP_TO_INR = 105.50
df["price_inr"] = df["price_gbp"] * GBP_TO_INR




/tmp/ipykernel_1871/1601590273.py:5: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df["price_gbp"].fillna(median_price, inplace=True)


# (4) SQLite Schema


In [4]:
conn = sqlite3.connect("books.db")
cur = conn.cursor()

cur.execute("""
CREATE TABLE IF NOT EXISTS categories (
    category_id INTEGER PRIMARY KEY,
    category_name TEXT UNIQUE
)
""")

cur.execute("""
CREATE TABLE IF NOT EXISTS books (
    book_id INTEGER PRIMARY KEY,
    title TEXT,
    price_gbp REAL,
    price_inr REAL,
    rating INTEGER,
    in_stock INTEGER,
    category_id INTEGER REFERENCES categories(category_id)
)
""")
conn.commit()


# (5) Insert Data

In [5]:
# Insert categories
for cat in df["category"].unique():
    cur.execute("INSERT OR IGNORE INTO categories(category_name) VALUES (?)", (cat,))
conn.commit()

# Insert books
for _, row in df.iterrows():
    cat_id = cur.execute("SELECT category_id FROM categories WHERE category_name=?",(row["category"],)).fetchone()[0]
    cur.execute("""
        INSERT INTO books(title, price_gbp, price_inr, rating, in_stock, category_id)
        VALUES (?, ?, ?, ?, ?, ?)
    """, (row["title"], row["price_gbp"], row["price_inr"], row["rating"], int(row["in_stock"]), cat_id))
conn.commit()


# (6) SQL Queries

In [6]:
# Example queries
print(pd.read_sql("SELECT DISTINCT rating FROM books", conn))
print(pd.read_sql("SELECT * FROM books WHERE rating=5 ORDER BY price_inr DESC LIMIT 10", conn))
print(pd.read_sql("SELECT category_name, COUNT(*) FROM books JOIN categories USING(category_id) GROUP BY category_name", conn))
print(pd.read_sql("SELECT * FROM books WHERE price_gbp BETWEEN 20 AND 30", conn))  # IN/BETWEEN example


   rating
0       2
1       4
2       3
3       1
4       5
   book_id                                              title price_gbp  \
0       11                 1,000 Places to See Before You Die      None   
1       20             A Time of Torment (Charlie Parker #14)      None   
2       29  What Happened on Beale Street (Secrets of the ...      None   
3       30  The Bachelor Girl's Guide to Murder (Herringfo...      None   
4       34                  The Silkworm (Cormoran Strike #2)      None   
5       40                                  The Girl You Lost      None   
6       46            A Flight of Arrows (The Pathfinders #2)      None   
7       48                                       Mrs. Houdini      None   
8       57                              The Passion of Dolssa      None   
9       59                             Voyager (Outlander #3)      None   

  price_inr  rating  in_stock  category_id  
0      None       5         1            1  
1      None       5     

# (7) Pandas Merge Equivalence

In [7]:
books_df = pd.read_sql("SELECT * FROM books", conn)
cats_df = pd.read_sql("SELECT * FROM categories", conn)

merged = pd.merge(books_df, cats_df, on="category_id")
merged.head()


,book_id,title,price_gbp,price_inr,rating,in_stock,category_id,category_name
0,1,It's Only the Himalayas,None,None,2,1,1,Travel
1,2,Full Moon over Noahâs Ark: An Odyssey to Mou...,None,None,4,1,1,Travel
2,3,See America: A Celebration of Our National Par...,None,None,3,1,1,Travel
3,4,Vagabonding: An Uncommon Guide to the Art of L...,None,None,2,1,1,Travel
4,5,Under the Tuscan Sun,None,None,3,1,1,Travel
